# 🗺️ Semana 2 · Unidad 1 — Diseño de Algoritmos

## Información del Curso

| Aspecto | Detalle |
|--------|--------|
| **Universidad** | Universidad de Talca, Chile |
| **Carrera** | Ingeniería Civil en Informática |
| **Semestre** | 2°-3° año |
| **Curso** | Algoritmos y Estructuras de Datos |
| **Docente** | PhD. César Astudillo |
| **Clase** | Semana 2 · Unidad 1 — Diseño de algoritmos |
| **Duración** | 90 minutos |

---
> 🎯 *Este notebook está diseñado para ser ejecutado en clase de forma interactiva.*
> *Ejecuta las celdas en orden de arriba hacia abajo.*

> 📌 **Cómo está armada esta clase.** No empezamos con una tabla de paradigmas. Empezamos
> con **un problema**, vemos qué idea lo resuelve, y recién al final le ponemos nombre a la
> idea. Un paradigma se entiende cuando ya lo viste funcionar, no antes.

In [ ]:
# Verificación de dependencias — ejecutar primero
import sys
required = {'numpy': 'numpy', 'matplotlib': 'matplotlib'}
for nombre, paquete in required.items():
    try:
        __import__(paquete)
        print(f"✅ {nombre} instalado correctamente")
    except ImportError:
        print(f"❌ {nombre} NO encontrado — instala con: pip install {paquete}")
print("\n🐍 Python", sys.version.split()[0], "| Todo listo para comenzar.")

## 🎯 Objetivos de Aprendizaje

Al finalizar esta sesión, el estudiante será capaz de:

1. **Comprender** qué es un algoritmo en términos de entradas, proceso y salidas, y por qué
   dos algoritmos correctos para el mismo problema pueden diferir en escala de uso.
2. **Identificar** la idea central de cuatro estrategias de diseño: dividir para conquistar,
   codicioso, programación dinámica y backtracking.
3. **Implementar** una versión mínima de cada estrategia sobre un problema pequeño.
4. **Analizar** por qué una estrategia codiciosa puede fallar y qué hace la programación
   dinámica para arreglarlo.
5. **Resolver** el problema de decidir qué estrategia conviene ante un problema nuevo.

# Sección 1: Un algoritmo es una idea, no un programa (12 minutos)

> 📌 **Definición:** un **algoritmo** es un procedimiento finito y no ambiguo que, a partir
> de unas **entradas**, ejecuta un **proceso** y produce unas **salidas**, resolviendo todas
> las instancias de un problema.

Fíjate en lo que *no* dice la definición: no dice Python, no dice computador. El algoritmo
es la idea; el programa es una forma de escribirla.

## El problema de hoy

Calcular $x^n$ para un entero $n \ge 0$. Suena trivial. Vamos a resolverlo dos veces.

**Primera idea — la definición literal:** $x^n = x \cdot x \cdots x$, $n$ veces.

**Segunda idea — partir el exponente por la mitad:**

$$x^n = \begin{cases} (x^{n/2})^2 & \text{si } n \text{ es par}\\[2pt] x \cdot x^{n-1} & \text{si } n \text{ es impar}\end{cases}$$

Las dos son correctas. Las dos dan exactamente el mismo número. Pero no sirven para lo mismo.

> 🎙️ **[PAUSA PROFESOR]** Pregunta sugerida: *"¿Cuántas multiplicaciones hace cada una para
> n = 1000? Estimen antes de ejecutar la celda."*

In [ ]:
def potencia_lenta(x, n):
    """
    Calcula x^n multiplicando n veces. La definición literal.

    Complejidad:
        Temporal: O(n) multiplicaciones
        Espacial: O(1)
    """
    resultado = 1
    for _ in range(n):
        resultado *= x
    return resultado


def potencia_rapida(x, n):
    """
    Calcula x^n partiendo el exponente por la mitad en cada paso.

    Si n es par:    x^n = (x^(n/2))^2   -> una multiplicación y medio problema
    Si n es impar:  x^n = x * x^(n-1)   -> una multiplicación y n pasa a ser par

    Complejidad:
        Temporal: O(log n) multiplicaciones — n se reduce a la mitad
        Espacial: O(log n) por la pila de recursión
    """
    if n == 0:
        return 1
    if n % 2 == 0:
        mitad = potencia_rapida(x, n // 2)
        return mitad * mitad
    return x * potencia_rapida(x, n - 1)


# Las dos dan lo mismo
print(f"potencia_lenta(2, 10)  = {potencia_lenta(2, 10)}")
print(f"potencia_rapida(2, 10) = {potencia_rapida(2, 10)}")
print(f"¿Coinciden en 0..30?     {all(potencia_lenta(3, k) == potencia_rapida(3, k) for k in range(31))}")

## Contemos las multiplicaciones

No midamos tiempo todavía: contemos **operaciones**. El tiempo depende del computador; el
número de operaciones depende de la idea.

In [ ]:
def contar_lenta(n):
    """Multiplicaciones que hace potencia_lenta."""
    return n


def contar_rapida(n):
    """Multiplicaciones que hace potencia_rapida."""
    if n == 0:
        return 0
    if n % 2 == 0:
        return contar_rapida(n // 2) + 1
    return contar_rapida(n - 1) + 1


print(f"{'n':>10} {'lenta':>12} {'rápida':>10} {'veces menos':>14}")
print("-" * 50)
for n in [10, 100, 1000, 10**6, 10**9]:
    a, b = contar_lenta(n), contar_rapida(n)
    print(f"{n:>10} {a:>12} {b:>10} {a // b:>13}x")

print("\n👉 Para n = mil millones: 1.000.000.000 multiplicaciones contra 42.")
print("   No cambiamos de computador. Cambiamos de idea.")

## La traza: por qué son 44 y no mil millones

Sigamos el exponente. Es la única variable que importa.

In [ ]:
def traza_potencia(n):
    """Imprime cómo se reduce el exponente hasta llegar a 0."""
    print(f"{'paso':>5} {'n':>12}  qué hace")
    print("-" * 46)
    paso = 0
    while n > 0:
        if n % 2 == 0:
            print(f"{paso:>5} {n:>12}  par     -> elevo al cuadrado, n = {n // 2}")
            n //= 2
        else:
            print(f"{paso:>5} {n:>12}  impar   -> saco una x,        n = {n - 1}")
            n -= 1
        paso += 1
    print(f"{paso:>5} {0:>12}  listo")
    print(f"\nTotal: {paso} multiplicaciones.")


traza_potencia(1000)

> 💡 **Insight:** cada dos pasos, en el peor caso, el exponente se parte por la mitad. Un
> número que se parte por la mitad llega a cero en $\log_2 n$ pasos. Esa es toda la razón.

Acabas de ver la primera estrategia de diseño. Su nombre viene después; primero veamos
otras tres.

# Sección 2: Dividir para conquistar (15 minutos)

## La idea

> 📌 **La idea:** si un problema de tamaño $n$ se puede partir en subproblemas **del mismo
> tipo** y de tamaño menor, resuélvelos por separado y combina sus respuestas.

Tres pasos, siempre los mismos:

```
   dividir      el problema en partes más pequeñas del mismo tipo
   conquistar   cada parte, recursivamente
   combinar     las respuestas parciales en la respuesta final
```

Eso fue exactamente `potencia_rapida`: dividir $n$ en $n/2$, conquistar recursivamente,
combinar elevando al cuadrado.

## Por qué funciona: la recurrencia

El costo se escribe solo:

$$T(n) = \underbrace{a}_{\text{subproblemas}} \cdot\; T(n/b) \;+\; \underbrace{f(n)}_{\text{costo de dividir y combinar}}$$

| Algoritmo | Recurrencia | Resultado |
|---|---|---|
| Potencia rápida | $T(n) = T(n/2) + O(1)$ | $O(\log n)$ |
| Búsqueda binaria | $T(n) = T(n/2) + O(1)$ | $O(\log n)$ |
| Merge Sort *(semana 9)* | $T(n) = 2T(n/2) + O(n)$ | $O(n \log n)$ |

> ⚠️ **Importante:** dividir solo sirve si las partes son **independientes**. Si los
> subproblemas se repiten entre sí, dividir hace trabajo redundante — y eso nos lleva
> directo a la Sección 4.

Veámoslo en el caso más pequeño posible: buscar en un arreglo ordenado.

In [ ]:
def busqueda_binaria(arr, objetivo, verbose=False):
    """
    Busca `objetivo` en un arreglo ORDENADO, descartando la mitad en cada paso.

    Parámetros:
        arr (list): arreglo ordenado ascendentemente
        objetivo: valor a buscar
        verbose (bool): imprime la traza del intervalo de búsqueda

    Retorna:
        int: índice del objetivo, o -1 si no está

    Complejidad:
        Temporal: O(log n) — el intervalo se parte por la mitad cada vez
        Espacial: O(1)
    """
    lo, hi = 0, len(arr) - 1
    paso = 0
    if verbose:
        print(f"{'paso':>5} {'lo':>4} {'hi':>4} {'medio':>6} {'quedan':>7}  intervalo vivo")
        print("-" * 62)
    while lo <= hi:
        medio = (lo + hi) // 2
        if verbose:
            print(f"{paso:>5} {lo:>4} {hi:>4} {medio:>6} {hi-lo+1:>7}  {arr[lo:hi+1]}")
        if arr[medio] == objetivo:
            if verbose:
                print(f"\n✅ encontrado en el índice {medio} tras {paso+1} comparaciones")
            return medio
        if arr[medio] < objetivo:
            lo = medio + 1      # descarto la mitad izquierda, incluido el medio
        else:
            hi = medio - 1      # descarto la mitad derecha
        paso += 1
    if verbose:
        print(f"\n❌ no está; se hicieron {paso} comparaciones")
    return -1


datos = [2, 5, 8, 12, 16, 23, 38, 56, 72, 91, 99]
busqueda_binaria(datos, 99, verbose=True)

> 🎙️ **[PAUSA PROFESOR]** Pregunta sugerida: *"En un arreglo de un millón de elementos,
> ¿cuántas comparaciones hace esta búsqueda en el peor caso?"* (Respuesta: 20. $\log_2 10^6 \approx 20$.)

# Sección 3: Estrategia codiciosa (15 minutos)

## La idea

> 📌 **La idea:** construye la solución paso a paso, y en cada paso toma **la mejor opción
> disponible en ese momento**, sin volver atrás nunca.

No hay recursión, no hay subproblemas, no hay memoria. Es la estrategia más simple que
existe: en cada paso, agarra lo que más te conviene ahora.

## El problema: dar el vuelto con la menor cantidad de monedas

Tienes monedas de $500, $100, $50 y $10. Debes dar $t de vuelto usando el menor número de
monedas posible.

La estrategia codiciosa es la que usa cualquier cajero: **parte por la moneda más grande que
quepa**, y repite.

In [ ]:
def vuelto_codicioso(monto, monedas, verbose=False):
    """
    Da el vuelto tomando siempre la moneda más grande que quepa.

    Parámetros:
        monto (int): cantidad a devolver
        monedas (list): denominaciones disponibles (se ordenan de mayor a menor)
        verbose (bool): imprime la decisión de cada paso

    Retorna:
        list: monedas entregadas

    Complejidad:
        Temporal: O(monto / moneda_menor) en el peor caso
        Espacial: O(1) más la salida
    """
    entregadas = []
    restante = monto
    if verbose:
        print(f"{'paso':>5} {'restante':>10} {'elijo':>8}  razón")
        print("-" * 52)
    paso = 0
    for m in sorted(monedas, reverse=True):
        while restante >= m:
            if verbose:
                print(f"{paso:>5} {restante:>10} {m:>8}  es la mayor que cabe en {restante}")
            entregadas.append(m)
            restante -= m
            paso += 1
    if verbose:
        print(f"\nTotal: {len(entregadas)} monedas -> {entregadas}")
    return entregadas


CLP = [500, 100, 50, 10]
vuelto_codicioso(1240, CLP, verbose=True)

## Dónde se cae: el punto que hay que entender

La estrategia codiciosa es rápida y simple. También es **frecuentemente incorrecta**.

Con las monedas chilenas funciona. Pero el algoritmo no sabe nada de Chile: si le cambias
las denominaciones, sigue haciendo lo mismo — y ahí falla.

Probemos con un sistema inventado: monedas de **1, 3 y 4**, para un monto de **6**.

> 🎙️ **[PAUSA PROFESOR]** Pregunta sugerida: *"¿Cuál es la mejor forma de dar 6 con monedas
> de 1, 3 y 4? ¿Qué va a hacer el algoritmo codicioso?"*

In [ ]:
RARO = [1, 3, 4]

print("=== Lo que hace el algoritmo codicioso ===")
codicioso = vuelto_codicioso(6, RARO, verbose=True)

print("\n=== Lo que era posible ===")
print("6 = 3 + 3  ->  2 monedas")

print(f"\nCodicioso: {len(codicioso)} monedas {codicioso}")
print(f"Óptimo:    2 monedas [3, 3]")
print("\n❌ El algoritmo codicioso NO es óptimo aquí.")
print("   Tomó el 4 porque era el mayor que cabía, y con eso se condenó:")
print("   le quedaron 2, que solo se pueden formar con dos monedas de 1.")

> ⚠️ **Importante:** que una estrategia codiciosa funcione para un conjunto de datos **no
> demuestra** que sea correcta. Hay que probar dos propiedades:
>
> 1. **Elección codiciosa:** existe una solución óptima que contiene la elección local.
> 2. **Subestructura óptima:** al fijar esa elección, lo que queda es el mismo problema, más pequeño.
>
> Con monedas 1/3/4 la primera propiedad **es falsa**: ninguna solución óptima de 6 usa el 4.

> 💡 **Insight:** lo valioso de esta estrategia no es implementarla — son cinco líneas — sino
> saber **cuándo tienes derecho a usarla**. Ese es el trabajo de diseño.

Entonces, ¿cómo se resuelve bien el caso 1/3/4? Se necesita una idea que **no se
comprometa** con una decisión antes de conocer sus consecuencias.

# Sección 4: Programación dinámica (18 minutos)

## La idea

> 📌 **La idea:** en vez de decidir a ciegas, resuelve **todos** los subproblemas pequeños,
> guarda sus respuestas, y construye la respuesta grande a partir de ellas.

La palabra clave es **subproblemas solapados**. Si al dividir el problema los pedazos se
repiten, resolverlos una y otra vez es un despilfarro: resuélvelos una vez y anótalos.

## El mismo problema del vuelto, ahora bien resuelto

Definamos la respuesta que buscamos:

$$M(t) = \text{mínimo número de monedas para formar exactamente } t$$

La relación que la define es directa: si la última moneda que entrego es $m$, entonces antes
tuve que formar $t - m$. Como no sé cuál es la última, **las pruebo todas y me quedo con la mejor**:

$$M(t) = 1 + \min_{m \,\le\, t} M(t - m), \qquad M(0) = 0$$

Fíjate en la diferencia con la estrategia codiciosa: aquella **elegía** una moneda; esta
**considera todas** y deja que el mínimo decida.

> 🎙️ **[PAUSA PROFESOR]** Pregunta sugerida: *"¿Cuántas veces necesitaríamos M(2) si
> calculáramos esto recursivamente sin guardar nada?"*

In [ ]:
def vuelto_dp(monto, monedas, verbose=False):
    """
    Mínimo número de monedas para formar `monto`, probando todas las opciones.

    Construye la tabla M[0..monto] de abajo hacia arriba: para calcular M[t]
    ya tenemos resueltos todos los M[t - m], porque t - m < t.

    Parámetros:
        monto (int): cantidad a formar
        monedas (list): denominaciones disponibles
        verbose (bool): imprime la tabla mientras se llena

    Retorna:
        (int, list): número mínimo de monedas y una combinación que lo logra

    Complejidad:
        Temporal: O(monto * len(monedas)) — una celda por monto, todas las monedas
        Espacial: O(monto) para la tabla
    """
    INF = float("inf")
    M = [0] + [INF] * monto        # M[t] = mínimo de monedas para formar t
    elegida = [None] * (monto + 1)  # para reconstruir la respuesta

    if verbose:
        print(f"{'t':>4} {'M[t]':>6}  se calcula probando")
        print("-" * 54)

    for t in range(1, monto + 1):
        opciones = []
        for m in monedas:
            if m <= t and M[t - m] + 1 < M[t]:
                M[t] = M[t - m] + 1
                elegida[t] = m
            if m <= t:
                opciones.append(f"M[{t-m}]+1={M[t-m]+1 if M[t-m] < INF else '∞'}")
        if verbose:
            print(f"{t:>4} {M[t]:>6}  {', '.join(opciones)}")

    # reconstruir una combinación óptima siguiendo las monedas elegidas
    combinacion, t = [], monto
    while t > 0 and elegida[t] is not None:
        combinacion.append(elegida[t])
        t -= elegida[t]

    return M[monto], combinacion


print("=== Tabla para monto 6 con monedas [1, 3, 4] ===\n")
n, comb = vuelto_dp(6, [1, 3, 4], verbose=True)
print(f"\n✅ Óptimo: {n} monedas -> {comb}")
print("\nCompara con el codicioso, que daba 3 monedas [4, 1, 1].")
print("La tabla no adivinó: probó todas las últimas monedas posibles y se quedó con la mejor.")

## Los dos ingredientes

Para que una programación dinámica exista hacen falta dos cosas:

| Ingrediente | Qué significa | En el vuelto |
|---|---|---|
| **Subestructura óptima** | La solución óptima se arma con soluciones óptimas de subproblemas | $M(t)$ usa el mejor $M(t-m)$ |
| **Subproblemas solapados** | Los mismos subproblemas aparecen muchas veces | $M(2)$ se necesita para $M(3)$, $M(5)$, $M(6)$… |

> 💡 **Insight:** dividir para conquistar y programación dinámica parten del mismo lugar
> —romper el problema en subproblemas— y se separan en una sola pregunta: **¿los
> subproblemas se repiten?** Si no se repiten, divide. Si se repiten, tabula.

Comprobémoslo: sin tabla, la recursión repite trabajo de forma explosiva.

In [ ]:
llamadas = {"con_tabla": 0, "sin_tabla": 0}

def vuelto_recursivo(t, monedas):
    """Misma recurrencia, SIN guardar nada. Recalcula lo mismo una y otra vez."""
    llamadas["sin_tabla"] += 1
    if t == 0:
        return 0
    mejor = float("inf")
    for m in monedas:
        if m <= t:
            mejor = min(mejor, vuelto_recursivo(t - m, monedas) + 1)
    return mejor


print(f"{'monto':>7} {'sin tabla':>12} {'con tabla':>12} {'veces más':>12}")
print("-" * 48)
for t in [6, 10, 14, 18, 22]:
    llamadas["sin_tabla"] = 0
    vuelto_recursivo(t, [1, 3, 4])
    sin = llamadas["sin_tabla"]
    con = t * 3          # la tabla hace monto x len(monedas) trabajos
    print(f"{t:>7} {sin:>12} {con:>12} {sin // max(con, 1):>11}x")

print("\n👉 La recursión sin tabla crece exponencialmente; la tabla crece linealmente.")
print("   Misma idea, misma recurrencia. La única diferencia es anotar lo ya resuelto.")

# Sección 5: Backtracking (15 minutos)

## La idea

> 📌 **La idea:** construye la solución **una decisión a la vez**. Si en algún punto la
> solución parcial ya es inviable, **deshaz la última decisión** y prueba otra.

Es búsqueda exhaustiva, pero con la disciplina de abandonar temprano las ramas muertas.
Todo backtracking tiene la misma forma de tres líneas:

```
elegir      una opción y agregarla a la solución parcial
explorar    recursivamente lo que queda
deshacer    la elección  <- este paso es el que le da el nombre
```

> ⚠️ **Importante:** el paso de deshacer no es un detalle de implementación. Es lo que
> permite que la **misma** estructura de datos sirva para explorar todas las ramas, sin
> copiarla en cada llamada.

## El esqueleto, en ocho líneas

No vamos a resolver Sudoku ni colorear grafos hoy. Vamos a ver el esqueleto sobre el
problema más pequeño que lo necesita: **generar todos los subconjuntos** de un conjunto.

En cada elemento hay exactamente dos decisiones: lo incluyo o no lo incluyo.

In [ ]:
def subconjuntos(elementos):
    """
    Genera todos los subconjuntos, decidiendo elemento por elemento.

    Complejidad:
        Temporal: O(2^n) — hay 2^n subconjuntos y cada uno se construye una vez
        Espacial: O(n) de pila y de solución parcial
    """
    resultado = []
    parcial = []

    def explorar(i):
        if i == len(elementos):          # ya decidí sobre todos
            resultado.append(list(parcial))
            return
        # decisión A: NO incluir elementos[i]
        explorar(i + 1)
        # decisión B: SÍ incluirlo
        parcial.append(elementos[i])     # elegir
        explorar(i + 1)                  # explorar
        parcial.pop()                    # deshacer  <- el backtrack

    explorar(0)
    return resultado


for s in subconjuntos(["a", "b", "c"]):
    print(s)
print(f"\n{len(subconjuntos(['a','b','c']))} subconjuntos para 3 elementos = 2^3")

## La traza: ver el «deshacer» ocurriendo

Esto es lo único que hay que entender de backtracking. Sigamos la solución parcial.

In [ ]:
def subconjuntos_traza(elementos):
    """Imprime cada elegir / explorar / deshacer, con sangría por profundidad."""
    parcial = []
    encontrados = []

    def explorar(i):
        sangria = "  " * i
        if i == len(elementos):
            print(f"{sangria}✅ solución: {parcial if parcial else '{}'}")
            encontrados.append(list(parcial))
            return
        e = elementos[i]
        print(f"{sangria}· decido sobre {e!r}, parcial = {parcial}")
        explorar(i + 1)
        parcial.append(e)
        print(f"{sangria}  elijo {e!r}      -> parcial = {parcial}")
        explorar(i + 1)
        quitado = parcial.pop()
        print(f"{sangria}  DESHAGO {quitado!r}   -> parcial = {parcial}")

    explorar(0)
    print(f"\nTotal: {len(encontrados)} subconjuntos.")


subconjuntos_traza(["a", "b"])

> 🎙️ **[PAUSA PROFESOR]** Pregunta sugerida: *"Si en vez de todos los subconjuntos
> quisiéramos solo los que suman menos de 10, ¿dónde agregarían la poda? ¿Qué se ganaría?"*

> 💡 **Insight:** cuando a este esqueleto le agregas una prueba que corta ramas inviables,
> tienes **poda**. Y si además usas una **cota** para descartar ramas que sí son viables pero
> no pueden mejorar la mejor solución hallada, tienes **ramificación y poda**. Es la misma
> idea, cada vez más informada.

Ramificación y poda queda fuera de esta clase: está en el material de profundización.

# Sección 6: Ahora sí, cómo elegir (10 minutos)

Recién ahora la tabla significa algo, porque cada fila es algo que ya viste funcionar.

| Estrategia | La idea en una frase | Señal de que aplica | Ejemplo de hoy |
|---|---|---|---|
| **Dividir para conquistar** | Parte, resuelve las partes, combina | Los subproblemas son del mismo tipo y **no se repiten** | Potencia rápida, búsqueda binaria |
| **Codicioso** | Toma lo mejor de cada paso y no mires atrás | Puedes **demostrar** que la elección local es segura | Vuelto con monedas chilenas |
| **Programación dinámica** | Resuelve y anota todos los subproblemas | Los subproblemas **se repiten** y lo local no basta | Vuelto con monedas 1/3/4 |
| **Backtracking** | Decide, explora, deshaz | Hay que **construir** una solución bajo restricciones | Todos los subconjuntos |

## Las dos preguntas que deciden

Casi siempre basta con responder dos cosas:

```
1. ¿Puedo partir el problema en subproblemas del mismo tipo?

   NO  -> ¿necesito construir la solución probando combinaciones?
          SÍ -> BACKTRACKING
          NO -> probablemente sea un recorrido o una búsqueda directa

   SÍ  -> 2. ¿Los subproblemas SE REPITEN?

          NO -> DIVIDIR PARA CONQUISTAR
          SÍ -> ¿me basta con la mejor decisión local?
                SÍ, y puedo demostrarlo -> CODICIOSO
                NO                      -> PROGRAMACIÓN DINÁMICA
```

> ⚠️ **Importante:** «puedo demostrarlo» no es un adorno. El caso 1/3/4 de la Sección 3 es
> exactamente lo que pasa cuando uno se salta esa demostración: el código corre, entrega un
> resultado, y el resultado está mal.

> 🎙️ **[PAUSA PROFESOR]** Ejercicio rápido: *"¿Dónde cae Merge Sort en este esquema? ¿Y
> Fibonacci recursivo?"* — Merge Sort: parte, no se repiten → dividir. Fibonacci: parte, y
> `fib(n-2)` se repite muchísimo → programación dinámica.

## 🧪 Ejercicio 1: Multiplicación del campesino ruso ⭐

**Descripción:** la misma idea de `potencia_rapida`, pero con sumas. Para multiplicar
$a \times b$ sin usar el operador `*`:

- si $b$ es par: $a \times b = (a + a) \times (b/2)$
- si $b$ es impar: $a \times b = a + a \times (b-1)$

**Entrada:** dos enteros `a` y `b`, con $b \ge 0$.
**Salida:** el producto `a * b`.

**Ejemplo:**
```
Entrada: a=13, b=11
Salida:  143
```

**Restricciones:** no uses `*` entre `a` y `b`. Solo sumas, `//` y comparaciones.
**Complejidad esperada:** O(log b) sumas

In [ ]:
def multiplicacion_rusa(a, b):
    """
    Multiplica a por b usando solo sumas, partiendo b por la mitad.

    Parámetros:
        a (int): primer factor
        b (int): segundo factor, no negativo
    Retorna:
        int: el producto a*b
    """
    # Tu código aquí
    pass

In [ ]:
def verificar_ejercicio_1(fn):
    """Ejecuta casos de prueba para el ejercicio 1."""
    import time
    casos = [
        ((3, 5), 15, "caso normal"),
        ((13, 11), 143, "ambos impares"),
        ((7, 8), 56, "segundo factor par"),
        ((1, 100), 100, "primer factor 1"),
        ((0, 50), 0, "primer factor 0"),
        ((99, 0), 0, "segundo factor 0"),
        ((12345, 6789), 12345 * 6789, "números grandes: debe ser O(log b)"),
    ]
    aprobados = 0
    for (a, b), esperado, desc in casos:
        t0 = time.perf_counter()
        try:
            r = fn(a, b)
            t1 = time.perf_counter()
            if r == esperado:
                print(f"  ✅ {desc} ({(t1-t0)*1000:.2f}ms) — {a}x{b} = {r}")
                aprobados += 1
            else:
                print(f"  ❌ {desc}\n     Esperado: {esperado}\n     Obtenido: {r}")
        except Exception as e:
            print(f"  💥 {desc} — Error: {e}")
    print(f"\n{'🎉 Todos los casos pasaron!' if aprobados == len(casos) else f'⚠️  {aprobados}/{len(casos)} casos correctos'}")

verificar_ejercicio_1(multiplicacion_rusa)

In [ ]:
# ═══════════════════════════════════════════════════
# SOLUCIÓN — Descomenta para ver después de intentarlo
# ═══════════════════════════════════════════════════

# def multiplicacion_rusa(a, b):
#     """Misma estructura que potencia_rapida, cambiando producto por suma."""
#     # Paso 1: caso base — multiplicar por 0 da 0
#     if b == 0:
#         return 0
#     # Paso 2: b par -> duplico a y reduzco b a la mitad
#     if b % 2 == 0:
#         return multiplicacion_rusa(a + a, b // 2)
#     # Paso 3: b impar -> saco una copia de a y dejo b par
#     return a + multiplicacion_rusa(a, b - 1)
#     # Complejidad: O(log b) sumas, O(log b) de pila

## 🧪 Ejercicio 2: ¿Es seguro ser codicioso? ⭐⭐

**Descripción:** escribe una función que, dado un sistema de monedas y un monto máximo,
encuentre **el monto más pequeño** en el que la estrategia codiciosa falla, es decir, donde
usa más monedas que el óptimo.

Ya tienes `vuelto_codicioso` y `vuelto_dp`: úsalas.

**Entrada:**
- `monedas` (list): denominaciones, incluye siempre el 1
- `tope` (int): hasta qué monto revisar

**Salida:** el menor monto donde codicioso ≠ óptimo, o `None` si coinciden en todo el rango.

**Ejemplo:**
```
Entrada: monedas=[1, 3, 4], tope=20
Salida:  6      (codicioso da 4+1+1 = 3 monedas; el óptimo es 3+3 = 2)

Entrada: monedas=[1, 5, 10, 25], tope=100
Salida:  None   (el sistema de EE.UU. es seguro para el codicioso)
```

**Restricciones:** $1 \le$ tope $\le 500$.
**Complejidad esperada:** O(tope² · |monedas|)

> 💡 **Pista:** para cada monto de 1 a `tope`, compara `len(vuelto_codicioso(...))` con el
> primer valor que devuelve `vuelto_dp(...)`. El primero que difiera es la respuesta.

In [ ]:
def primer_fallo_codicioso(monedas, tope):
    """
    Encuentra el menor monto donde la estrategia codiciosa no es óptima.

    Parámetros:
        monedas (list): denominaciones disponibles
        tope (int): monto máximo a revisar
    Retorna:
        int | None: el monto donde falla, o None si nunca falla
    """
    # Tu código aquí
    pass

In [ ]:
def verificar_ejercicio_2(fn):
    """Ejecuta casos de prueba para el ejercicio 2."""
    import time
    casos = [
        (([1, 3, 4], 20), 6, "el contraejemplo de la clase"),
        (([1, 5, 10, 25], 100), None, "monedas de EE.UU.: el codicioso es seguro"),
        (([10, 50, 100, 500, 1], 1000), None, "monedas chilenas: seguro"),
        (([1, 4, 5], 20), 8, "4+4 gana a 5+1+1+1"),
        (([1, 2], 30), None, "sistema trivial"),
        (([1, 6, 9], 30), 12, "9+1+1+1 pierde contra 6+6"),
    ]
    aprobados = 0
    for (monedas, tope), esperado, desc in casos:
        t0 = time.perf_counter()
        try:
            r = fn(list(monedas), tope)
            t1 = time.perf_counter()
            if r == esperado:
                print(f"  ✅ {desc} ({(t1-t0)*1000:.1f}ms) — {r}")
                aprobados += 1
            else:
                print(f"  ❌ {desc}\n     Esperado: {esperado}\n     Obtenido: {r}")
        except Exception as e:
            print(f"  💥 {desc} — Error: {e}")
    print(f"\n{'🎉 Todos los casos pasaron!' if aprobados == len(casos) else f'⚠️  {aprobados}/{len(casos)} casos correctos'}")

verificar_ejercicio_2(primer_fallo_codicioso)

In [ ]:
# ═══════════════════════════════════════════════════
# SOLUCIÓN — Descomenta para ver después de intentarlo
# ═══════════════════════════════════════════════════

# def primer_fallo_codicioso(monedas, tope):
#     """Compara ambas estrategias monto por monto y reporta la primera diferencia."""
#     # Paso 1: recorrer los montos EN ORDEN, para que el primero hallado sea el menor
#     for t in range(1, tope + 1):
#         # Paso 2: cuántas monedas usa cada estrategia
#         n_codicioso = len(vuelto_codicioso(t, monedas))
#         n_optimo, _ = vuelto_dp(t, monedas)
#         # Paso 3: la primera discrepancia es la respuesta
#         if n_codicioso != n_optimo:
#             return t
#     return None
#     # Complejidad: O(tope^2 * |monedas|) temporal — vuelto_dp es O(tope*|monedas|)
#     #              y se llama tope veces

## 🔬 Zona de Experimentación

Las siguientes celdas son tuyas para experimentar. Algunas sugerencias:
- ¿Cuál es el sistema de monedas más pequeño (menos denominaciones) donde el codicioso falla?
- Modifica `subconjuntos` para que solo genere los subconjuntos de tamaño exactamente $k$.
  ¿Dónde pusiste la poda?
- Escribe `fibonacci` de las dos formas —recursiva pura y con tabla— y cuenta las llamadas.

In [ ]:
# Espacio libre para experimentar
# Sugerencia: prueba primer_fallo_codicioso con [1, 7, 10] y con [1, 2, 5, 10].


In [ ]:
# Espacio libre para experimentar
# Sugerencia: fibonacci con y sin tabla; cuenta las llamadas de cada versión.


## ✍️ Autoevaluación

**1. ¿Cuál es la única pregunta que separa «dividir para conquistar» de «programación dinámica»?**

- a) Si el problema es de optimización o no
- b) Si los subproblemas se solapan (se repiten) o no
- c) Si la solución es recursiva o iterativa
- d) Si el arreglo está ordenado

<details>
<summary>Ver respuesta</summary>

**b) Si los subproblemas se solapan.** Ambas parten el problema en subproblemas del mismo
tipo. Si los subproblemas son independientes, basta con dividir y combinar. Si se repiten,
resolverlos por separado es trabajo redundante y conviene tabular. La opción c) es un error
común: la programación dinámica se puede escribir recursivamente (con memoización) y dividir
para conquistar se puede escribir iterativamente.

</details>

---

**2. El algoritmo codicioso da 3 monedas para el monto 6 con denominaciones [1, 3, 4], cuando el óptimo son 2. ¿Por qué falló?**

- a) Porque el código tiene un error de implementación
- b) Porque las monedas no estaban ordenadas
- c) Porque no se cumple la propiedad de elección codiciosa: ninguna solución óptima usa el 4
- d) Porque 6 es un número par

<details>
<summary>Ver respuesta</summary>

**c) No se cumple la propiedad de elección codiciosa.** El algoritmo está bien implementado
y hace exactamente lo que promete: tomar la moneda mayor que quepa. El problema es que en
este sistema de monedas esa elección no es segura — tomar el 4 deja un resto de 2 que solo
se forma con dos monedas de 1. Que un algoritmo codicioso corra sin errores no dice nada
sobre si es correcto.

</details>

---

**3. En el esqueleto de backtracking, ¿para qué sirve el paso de «deshacer»?**

- a) Para liberar memoria
- b) Para que la misma estructura parcial sirva al explorar la siguiente rama
- c) Para ordenar las soluciones encontradas
- d) Es opcional, se puede omitir

<details>
<summary>Ver respuesta</summary>

**b) Para reutilizar la estructura parcial.** Sin el `pop()`, la solución parcial arrastraría
las decisiones de la rama anterior y las ramas siguientes explorarían estados incorrectos.
La alternativa sería copiar la lista completa en cada llamada, que es correcto pero mucho
más caro. No es opcional: omitirlo produce respuestas equivocadas, no solo lentitud.

</details>

---

**4. `potencia_rapida(x, n)` hace 42 multiplicaciones para n = mil millones, contra mil millones de la versión ingenua. ¿De dónde sale el 42?**

- a) De que el computador es más rápido
- b) De que el exponente se parte por la mitad, y $\log_2(10^9) \approx 30$
- c) De que Python optimiza la recursión
- d) De que se usan menos variables

<details>
<summary>Ver respuesta</summary>

**b) Del logaritmo.** Cada paso par divide el exponente por dos; los pasos impares solo
restan uno y dejan el número par, así que en el peor caso hay a lo más dos pasos por cada
bit del exponente: $2 \log_2 n$. Para $n = 10^9$ eso da a lo más unos 60, y en este caso
concreto son 42. La ganancia viene de la idea, no de la máquina.

</details>

---

**5. Quieres generar todas las formas de sentar a 8 personas en 8 sillas cumpliendo ciertas restricciones. ¿Qué estrategia usarías?**

- a) Dividir para conquistar
- b) Codicioso
- c) Programación dinámica
- d) Backtracking

<details>
<summary>Ver respuesta</summary>

**d) Backtracking.** Hay que **construir** una solución tomando decisiones sucesivas (quién
se sienta en cada silla) bajo restricciones, y hay que poder abandonar una asignación
parcial en cuanto viole una restricción. Es exactamente el patrón elegir / explorar /
deshacer. Es también el esquema de las N-Reinas y del Sudoku.

</details>

# Resumen

## Lo que aprendimos hoy

1. Un algoritmo es una **idea**, no un programa. Dos algoritmos correctos para el mismo
   problema pueden diferir en varios órdenes de magnitud: mil millones de operaciones
   contra 42.
2. **Dividir para conquistar:** parte en subproblemas independientes, resuelve, combina.
   El costo se lee de la recurrencia.
3. **Codicioso:** toma lo mejor de cada paso. Es la estrategia más simple y la más fácil de
   usar mal — hay que demostrar que la elección local es segura.
4. **Programación dinámica:** cuando los subproblemas se repiten y lo local no basta,
   resuélvelos todos una vez y anótalos.
5. **Backtracking:** elegir, explorar, deshacer. Para construir soluciones bajo restricciones.
6. La elección entre estrategias se reduce casi siempre a dos preguntas: *¿se puede partir?*
   y *¿los pedazos se repiten?*

## Para la próxima clase

La semana 3 responde la pregunta que quedó abierta: cuando decimos «$O(\log n)$» o
«$O(n^2)$», ¿qué significa exactamente y cómo se calcula? Ahí formalizamos el análisis de
complejidad que hoy hicimos contando a mano.

## 📚 Lecturas Recomendadas y Práctica

### Textbooks

| Libro | Edición | Capítulo | Tema |
|-------|---------|----------|------|
| Cormen et al. (CLRS) — *Introduction to Algorithms* | 4ª ed. | Cap. 1–2 | Qué es un algoritmo; el ejemplo de insertion sort |
| Cormen et al. (CLRS) — *Introduction to Algorithms* | 4ª ed. | Cap. 4 | Dividir para conquistar y recurrencias |
| Cormen et al. (CLRS) — *Introduction to Algorithms* | 4ª ed. | Cap. 14–15 | Programación dinámica y algoritmos voraces |
| Bhargava (Grok) — *Grokking Algorithms* | 2ª ed. | Cap. 1, 3, 4 | Búsqueda binaria, recursión, divide y vencerás |
| Bhargava (Grok) — *Grokking Algorithms* | 2ª ed. | Cap. 8, 9 | Voraces y programación dinámica, con dibujos |
| Goodrich, Tamassia & Goldwasser (GTG) — *Data Structures and Algorithms in Python* | 1ª ed. | Cap. 4 | Recursión |

### Material de profundización del curso

Un notebook por estrategia, para trabajo autónomo:
[`material_detallado/`](material_detallado/) — divide y vencerás, voraces, programación
dinámica, backtracking, y ramificación y poda.

### Recursos gratuitos en línea

- 🌐 [VisuAlgo — Recursion Tree](https://visualgo.net/en/recursion) — el árbol de llamadas de una recursión.
- 🎬 [Algorithms, Part I — Robert Sedgewick (Princeton)](https://www.coursera.org/learn/algorithms-part1) — el curso de referencia de este ramo.

### Práctica en Codeforces (soporta Python 3)

> 🔍 **Cómo filtrar:** ve a [codeforces.com/problemset](https://codeforces.com/problemset),
> escribe `greedy`, `dp` o `divide and conquer` en **Tags** y ajusta **Rating**.

**Escala de dificultad orientativa para este curso:**

| Rating | Nivel | Descripción |
|--------|-------|-------------|
| 800 | ⭐ | Aplicación directa — la mayoría puede resolverlo |
| 1000–1200 | ⭐⭐ | Requiere una pequeña adaptación |
| 1300+ | ⭐⭐⭐ | Combina la idea con otra — desafío |

**Problemas recomendados para este tópico:**

| # | Problema | Rating | Por qué es útil |
|---|----------|--------|-----------------|
| 1 | [4A — Watermelon](https://codeforces.com/problemset/problem/4/A) | ⭐ 800 | El primer problema de todos: entrada, proceso, salida |
| 2 | [158B — Taxi](https://codeforces.com/problemset/problem/158/B) | ⭐ 1100 | Estrategia codiciosa donde hay que justificar la elección |
| 3 | [231A — Team](https://codeforces.com/problemset/problem/231/A) | ⭐ 800 | Recorrido y conteo directo |
| 4 | [455A — Boredom](https://codeforces.com/problemset/problem/455/A) | ⭐⭐⭐ 1500 | Programación dinámica: el codicioso falla y hay que tabular |

⚠️ Los problemas 1 y 3 son el **mínimo esperado**. El 2 es intermedio y el 4 es desafío opcional.